### Start

In [1]:
%load_ext autoreload
%autoreload 2

from helpers import DATA_PATH, get_data_from_file

In [2]:
corrupt, clean = get_data_from_file('small')
len(corrupt)

64

### Space correction

In [59]:
from neuspell import BertChecker

checker = BertChecker(device='cuda')
checker.from_pretrained()

loading vocab from path:c:\users\brumda\documents\neuspell\neuspell\../neuspell_data/checkpoints/subwordbert-probwordnoise\vocab.pkl
initializing model
loading pretrained weights from path:c:\users\brumda\documents\neuspell\neuspell\../neuspell_data/checkpoints/subwordbert-probwordnoise
Loading model params from checkpoint dir: c:\users\brumda\documents\neuspell\neuspell\../neuspell_data/checkpoints/subwordbert-probwordnoise


In [91]:
def get_subtokens(tokens, start_index):
    """Get the first word after start index"""
    result = [tokens[start_index]]

    # Find indices of tokens after start_index that start with '##'
    mask = np.char.startswith(tokens[start_index + 1:], '##')

    # Find the first False (non-## token) in the mask
    # Else happens if the rest of the tokens are one word
    non_subtoken_indices = np.where(~mask)[0]
    end_idx = non_subtoken_indices[0] if len(non_subtoken_indices) > 0 else len(mask)

    result.extend(tokens[start_index + 1:start_index + 1 + end_idx])
    return np.array(result)

In [4]:
from transformers import BertTokenizerFast
import numpy as np

index = 378
tokenizer = BertTokenizerFast.from_pretrained("bert-base-cased")
tokenizer.do_basic_tokenize = True
tokenizer.tokenize_chinese_chars = False
orig_string = corrupt[index]
orig_string_clean = clean[index]
# test_string = '### General Contributition Guidelines'
# test_string_clean = '### General Contributition Guidelines'
pred_string = checker.correct_string(orig_string, correct_spaces=False)
# transformed_tokens = transformed_tokens[:14] + 's' + transformed_tokens[14:29] + 's' + transformed_tokens[29:]
print(f"input: {orig_string}", f"clean: {orig_string_clean}", f"output: {pred_string}", sep="\n")

IndexError: list index out of range

In [97]:
original_offsets = tokenizer(orig_string, return_offsets_mapping=True)['offset_mapping']
tokenization = np.array(tokenizer.tokenize(pred_string))
token_offsets = np.array(original_offsets[1:-1])  # first and last offsets are [CLS] and [SEP]
idx_offset = 0
out_idx = 0
in_row = 0

# Pre-allocate arrays
in_tok = np.array(tokenizer.tokenize(orig_string))
max_tokens = max(len(tokenization), len(original_offsets))
pretok_sent = np.empty(max_tokens, dtype=object)
offsets_merged = np.empty((max_tokens, 2), dtype=int)

i = 0
delta = 0
old_offset = 0
while i < len(tokenization):
    token = tokenization[i]
    in_bounds = i + idx_offset < len(token_offsets)
    if token.startswith("##"):
        # Continuation of previous token
        in_row += 1
        # merge
        pretok_sent[out_idx - 1] = pretok_sent[out_idx - 1] + token[2:]
        if in_bounds:
            old_offset = offsets_merged[out_idx - 1, 1]
            offsets_merged[out_idx - 1, 1] = token_offsets[i + idx_offset, 1]
            # print(old_offset, offsets_merged[out_idx - 1])
        else:
            print("out of bounds")

        # print(f"SubToken starting: {token}")
    else:
        if in_row >= 1:
            # check if the previous merged token has correct offsets
            start = i + idx_offset - in_row - 1 + delta
            word = get_subtokens(in_tok, start)
            # print(f"Fixed token: {pretok_sent[out_idx - 1]}")
            # print(f"Next Token: {token}")
            # print(f"Word: {word}")
            orig_toks = tokenizer.tokenize(''.join(tok.replace('##', '') for tok in word))
            # print(f"Orig_toks: {orig_toks}")

            if len(orig_toks) > in_row + 1:
                # print("changing offsets")
                # fix the offsets
                for _ in range(len(orig_toks) - (in_row + 1)):
                    offsets_merged[out_idx - 1, 1] = token_offsets[i + idx_offset, 1]
                    idx_offset += 1
            elif len(orig_toks) < in_row + 1:
                # print("new is more than orig")
                # print(len(orig_toks), in_row + 1)
                new_delta = len(orig_toks) - (in_row + 1)
                delta += new_delta
                offsets_merged[out_idx - 1, 1] = old_offset
                idx_offset += new_delta
                # print(delta)

            # print(80 * "-")

            in_row = 0

        # handle the new token
        pretok_sent[out_idx] = token

        if in_bounds:
            offsets_merged[out_idx] = token_offsets[i + idx_offset]
        else:
            prev_end = offsets_merged[out_idx - 1, 1]
            token_len = len(token)
            offsets_merged[out_idx] = (prev_end + 1, prev_end + 1 + token_len)
        out_idx += 1
    i += 1

# Truncate arrays to actual size
pretok_sent = pretok_sent[:out_idx]
offsets_merged = offsets_merged[:out_idx]

next_starts = np.roll(offsets_merged[:, 0], -1)[:-1]
current_ends = offsets_merged[:-1, 1]
mask = next_starts > current_ends
# boolean mask checking if the offsets align or next start index is greater than previous end
# indicating space in the original text
mask = np.append(mask, False)
tokens_arr = np.array(pretok_sent)
# add spaces back at corresponding places
tokens_with_space = np.where(mask, np.char.add(pretok_sent, ' '), pretok_sent)
reconstructed_text = "".join(tokens_with_space)

In [98]:
print(*original_offsets[1:-1])
print(in_tok.astype(str).tolist())
print(tokenizer.tokenize(pred_string))
print(160 * "-")
print(pretok_sent.astype(str).tolist())
print(*offsets_merged)
print(160 * "-")
print(mask)
print(160 * "-")
reconstructed_text

(0, 1) (1, 2) (2, 3) (4, 11) (12, 15) (15, 18) (18, 21) (21, 26) (27, 32) (32, 37) (37, 38)
['#', '#', '#', 'General', 'Con', '##tri', '##but', '##ition', 'Guide', '##lines', ':']
['#', '#', '#', 'General', 'Con', '##tribution', 'Guide', '##lines', ':']
----------------------------------------------------------------------------------------------------------------------------------------------------------------
['#', '#', '#', 'General', 'Contribution', 'Guidelines', ':']
[0 1] [1 2] [2 3] [ 4 11] [12 26] [27 37] [37 38]
----------------------------------------------------------------------------------------------------------------------------------------------------------------
[False False  True  True  True False False]
----------------------------------------------------------------------------------------------------------------------------------------------------------------


'### General Contribution Guidelines:'

In [99]:
print(orig_string)
print(80 * "-")
print(orig_string_clean)
print(80 * "-")
print(pred_string)

### General Contributition Guidelines:
--------------------------------------------------------------------------------
### General Contribution Guidelines:
--------------------------------------------------------------------------------
# # # General Contribution Guidelines :


In [17]:
tokenizer.tokenize('Congratulations')

['Con', '##gratulations']

### Detect typo

In [5]:
from detect_typo_model import TypoDetectionModel

In [7]:
pred_model = TypoDetectionModel()
pred_model.load_model()

Model loaded successfully
Model loaded from detect_typo_models/best_model.pt


In [16]:
for i in range(20):
    text = corrupt[i]
    print(text, clean[i], sep='\n')
    print(pred_model.predict(text))
    print(80 * '-')

team_number = tr[1].p.span.contents[0]
team_number = tds[1].p.span.contents[0]
0.47374123334884644
--------------------------------------------------------------------------------
* The internal method that handles the pointer out event from the browser.
* The internal method that handles the pointer over event from the browser.
0.5091391801834106
--------------------------------------------------------------------------------
To understand what is in the `dockercfg` field, convert the secret data to a
To understand what is in the `.dockercfg` field, convert the secret data to a
0.8158062696456909
--------------------------------------------------------------------------------
number of blocks has been removed.  The rpc calls are deprecated and will either
number of blocks has been removed.  The RPC calls are deprecated and will either
0.8373557329177856
--------------------------------------------------------------------------------
will be those of the redirected or rewriten routes w

In [10]:
import pandas as pd

test_df = pd.read_csv(DATA_PATH + "test_prob_df.csv", dtype={0: str, 1: float})

In [11]:
pred_model.evaluate(test_df)

Test Loss: 0.0784 | Test MAE: 0.1694


{'dev_loss': 0.07842605434442387, 'mae': 0.16937337815761566}

In [34]:
from transformers import BertTokenizerFast
from neuspell.seq_modeling.helpers import merge_subtokens

tokenizer = BertTokenizerFast.from_pretrained("bert-base-cased")
tokenizer.do_basic_tokenize = True
tokenizer.tokenize_chinese_chars = False
corrupt_lines, clean_lines = get_data_from_file('test')
acc_sen, corr2corr, corr2incorr, incorr2corr, incorr2incorr = 0, 0, 0, 0, 0

idx = 4
for idx in range(len(corrupt_lines)):
    corrupt = corrupt_lines[idx]
    clean = clean_lines[idx]
    prediction = checker.correct_string(corrupt, correct_spaces=False)

    corrupt = merge_subtokens(tokenizer.tokenize(corrupt))
    clean = merge_subtokens(tokenizer.tokenize(clean))
    prediction = merge_subtokens(tokenizer.tokenize(prediction))
    # print(corrupt, clean, prediction, sep='\n')
    # print(80*'-')

    acc_sen += (prediction == clean)
    same_len = (len(corrupt) == len(clean) == len(prediction))
    if not same_len:
        for corrupt_token, clean_token, predict_token in zip(corrupt.split(), clean.split(), prediction.split()):
            if corrupt_token == clean_token and predict_token == clean_token:
                corr2corr += 1
            elif corrupt_token == clean_token and predict_token != clean_token:
                corr2incorr += 1
            elif corrupt_token != clean_token and predict_token == clean_token:
                incorr2corr += 1
            elif corrupt_token != clean_token and predict_token != clean_token:
                incorr2incorr += 1
print("Results:")
print(acc_sen, corr2corr, corr2incorr, incorr2corr, incorr2incorr)

Results:
11511 809056 45463 13702 242905


In [30]:
diff = 0
for corr, clean in zip(corrupt_lines, clean_lines):
    diff += (len(corr) == len(clean))


In [32]:
print (diff)

19070


In [33]:
len (corrupt_lines)

67886

### More models

In [20]:
from happytransformer import HappyTextToText

# Load T5 model for grammar/spelling correction
happy_tt = HappyTextToText("T5", "vennify/t5-base-grammar-correction")

# Example sentence with typos
input_text = "He go to skool everi day."

# Correct the sentence
output = happy_tt.generate_text(f"grammar: {input_text}").text
output_no_pref = happy_tt.generate_text(input_text).text
print(output, output_no_pref, sep='\n')


04/29/2025 12:23:24 - INFO - happytransformer.happy_transformer -   Using device: cuda:0
04/29/2025 12:23:24 - INFO - happytransformer.happy_transformer -   Moving model to cuda:0
04/29/2025 12:23:24 - INFO - happytransformer.happy_transformer -   Initializing a pipeline
Device set to use cuda:0


He goes to school every day..!
He goes to school every day. He goes to school every day.


In [40]:
pred_func = lambda model, data: model.generate_text(f"grammar: {data}").text
print(pred_func(happy_tt, corrupt[0]))

Team_number = tr[1].p.span.contents[0]


In [21]:
from transformers import pipeline

# Initialize the grammar correction pipeline
grammar_corrector = pipeline("text2text-generation", model="prithivida/grammar_error_correcter_v1")

output = grammar_corrector(f"grammar: {input_text}")[0]['generated_text']
output_no_pref = grammar_corrector(input_text)[0]['generated_text']
print(output, output_no_pref, sep='\n')
# print(grammar_corrector("He go to skool eve ry day.")[0]['generated_text'])
# Output: He goes to school every day.


Device set to use cuda:0


Grammar: He goes to school every day.
He goes to skool every day.


In [20]:
func =  lambda model, text: model(text)[0]['generated_text']
print(func(grammar_corrector, "He go to school every day."))

He goes to school every day.


In [7]:
param_size = sum(p.numel() * p.element_size() for p in grammar_corrector.model.parameters())
buffer_size = sum(b.numel() * b.element_size() for b in grammar_corrector.model.buffers())
(param_size + buffer_size) / 1024**2

850.3095703125

In [22]:
from transformers import pipeline

# Initialize the text-generation pipeline for text correction
corrector = pipeline("text2text-generation", "pszemraj/bart-base-grammar-synthesis")

# Example text to correct

# Correct the text using the text-generation pipeline
# corrected_text = corrector(raw_text)[0]["generated_text"]

output = corrector(f"grammar: {input_text}")[0]['generated_text']
output_no_pref = corrector(input_text)[0]['generated_text']
print(output, output_no_pref, sep='\n')

# Print the corrected text
# print(corrected_text)


Device set to use cuda:0


grammar: He goes to school every day.
He goes to school every day.


In [17]:
func =  lambda model, text: model(text)[0]['generated_text']
print(func(corrector, "He go to school every day."))

He gets to school every day.


In [9]:
param_size = sum(p.numel() * p.element_size() for p in corrector.model.parameters())
buffer_size = sum(b.numel() * b.element_size() for b in corrector.model.buffers())
(param_size + buffer_size) / 1024**2

532.0384254455566

In [23]:
from transformers import pipeline

corrector = pipeline("text2text-generation",model="oliverguhr/spelling-correction-english-base")

output = corrector(f"grammar: {input_text}")[0]['generated_text']
output_no_pref = corrector(input_text)[0]['generated_text']
print(output, output_no_pref, sep='\n')

# print(corrector("les do  comparsion")[0]['generated_text'])


Device set to use cuda:0


Grammar, 56. He go to school every day.
He got to school every day.


In [26]:
param_size = sum(p.numel() * p.element_size() for p in fix_spelling.model.parameters())
buffer_size = sum(b.numel() * b.element_size() for b in fix_spelling.model.buffers())
(param_size + buffer_size) / 1024**2

532.0384254455566

In [28]:
func =  lambda model, text: model(text)[0]['generated_text']
print(func(corrector, "les do  comparsion"))

Let's do a comparison.


In [24]:
from transformers import pipeline

# Load the text generation pipeline with the desired model
corrector = pipeline("text2text-generation", model="grammarly/coedit-large")

# Input text with grammatical errors
# input_text = 'Fix grammatical errors in this sentence: When I grow up, I start to understand what he said is quite right.'

output = corrector(f"grammar: {input_text}")[0]['generated_text']
output_no_pref = corrector(input_text)[0]['generated_text']
print(output, output_no_pref, sep='\n')
# Generate corrected text using the pipeline
# result = corrector(input_text)

# Print the corrected output
# edited_text = result[0]['generated_text']
# print(edited_text)


Device set to use cuda:0


He goes to school every day.
He goes to school every day.


In [31]:
func =  lambda model, text: model(text)[0]['generated_text']
print(func(corrector, "Fix grammatical errors in this sentence: When I grow up, I start to understand what he said is quite right."))

When I grow up, I will start to understand what he said is quite right.


In [32]:
"grammarly/coedit-large".replace("/", "-")

'grammarly-coedit-large'